## Stage 2 — Patching v2: intervening at the IDENTITY TOKEN (the finding 07 recipe)

v1 (notebook 11) came out zero by design: patching 1-3 heads at natural scale at the LAST token,
random questions (ceiling ~0.004). Dissecting the llm-opinions code (finding 07) shows the
output CAN be shifted through the same interface — the difference lies in 4 things. v2 fixes
all of them:

| aspect | v1 (zero) | v2 (this one) |
|---|---|---|
| Prompt shape | identity = a natural sentence | identity = a **1-token answer** in a demographic QA block (llm-opinions style) |
| Intervention position | last token (no propagation) | **the identity token position** — the change propagates to every token after it |
| Strength | 1-3 heads, 1x | **all 32 heads of L11** (+ variants), scale 1x as an instrument + 5x as a steering comparison |
| Questions | random | **top-20 per pair by largest WD between the REAL distributions** |

The main trick: prompts A and B are **identical except for the 2 demographic answer letter tokens** —
so grafting the activations at those 2 positions = replacing the identity at its source,
and the `predB` baseline = the "full swap" ceiling, which can be compared directly.

Conditions:

| condition | content | role |
|---|---|---|
| `patch_L11_all32` | all 32 heads of L11, α=1 | **main hypothesis** |
| `patch_L11H16` | the star head only, α=1 | is 1 head enough? |
| `patch_L11L18_all` | all heads of L11+L18, α=1 | 2 layers |
| `patch_L11_all32_x5` | α=5 (base + 5·(donor−base)) | steering-style comparison |
| `patch_randhead` | 1 fixed random head, α=1 | negative control |
| `self_patch` | donor = itself (subset) | sanity check (must be ~0) |

The score is unchanged: the WD shift of the prediction toward the **donor group's REAL distribution**
(patching as a fidelity instrument, not steering).


## Before running: Kaggle setup

1. **Accelerator**: GPU T4 x2. **Internet: On**. Attach `opinionqa_intersectional.csv`.
2. A session left over from a crash -> RESTART SESSION.
3. **Download when finished:** `patching_v2_rows.csv` + `patching_v2_summary.csv`
   from `/kaggle/working/stage2_patching_v2/` -> put them into
   `results/06_patching_v2/`.

Estimate: ~5,300 forward passes ≈ 1-1.5 hours total.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance, wilcoxon
from tqdm.auto import tqdm

sys.last_traceback = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  WARNING: GPU {d} is not empty -> RESTART SESSION first!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv not found.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
TYPES_RUN = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]
N_PAIRS = 12
N_QUESTIONS = 20      # per pair, the top ones by largest real WD
MAX_OPTIONS = 6       # options of the opinion question (letters A-F); the demographic QA may have more
MIN_SHARED_Q = 20
N_SELF_PATCH = 3

STAR_LAYER, STAR_HEAD = 11, 16
SECOND_LAYER = 18

OUT_DIR = "/kaggle/working/stage2_patching_v2"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Data: pairs + the questions with the largest real difference

Difference from v1: the questions per pair are not random but the **top-20 by WD(realA, realB)** —
the questions where the two groups REALLY disagree the most. This raises the
instrument's ceiling.


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
for ty in TYPES_RUN:
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    # unique values per component -> the list of demographic QA options
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=sorted({c for p in pairs for c in p}), pairs=pairs,
                    pair_questions=pair_questions, needed=needed,
                    v1_opts=v1_opts, v2_opts=v2_opts)
    top_wd = np.mean([real_wd(a, b, qk) for (a, b), qs in pair_questions.items() for qk in qs])
    print(f"[{ty}] {len(plan[ty]['cells'])} cells, {len(pairs)} pairs, unique baselines "
          f"{len(needed)}, mean real WD of the chosen questions: {top_wd:.3f}")


## 2. Demographic-QA prompt + identity token position

Identity enters as a **1-token answer** in two QA blocks (one per component),
then the opinion question. Prompts A vs B are identical except for those 2 letter tokens -> the
intervention position is exactly the same, and predB = the full-swap ceiling.


In [ ]:
ATTR_QA = {
    "RACExRELIG":       ("What is this survey respondent's race?",
                         "What is this survey respondent's religion?"),
    "RELIGxPOLPARTY":   ("What is this survey respondent's religion?",
                         "What is this survey respondent's political party affiliation?"),
    "AGExPOLPARTY":     ("What is this survey respondent's age group?",
                         "What is this survey respondent's political party affiliation?"),
}
DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    p = plan[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, p["v1_opts"], v1), "", demo_block(q2, p["v2_opts"], v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def identity_positions(prompt):
    """Positions of the answer letter tokens of the 2 demographic QA blocks (after the 1st & 2nd 'Answer:')."""
    positions = []
    search_from = 0
    for _ in range(2):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))  # +1 BOS
        positions.append(pos)
        search_from = idx + 1
    return positions

# validation: A vs B differ only at 2 letter tokens, same positions
ty0 = TYPES_RUN[0]
(A0, B0) = plan[ty0]["pairs"][0]
qk0 = plan[ty0]["pair_questions"][(A0, B0)][0]
pA, pB = build_prompt(ty0, A0, qk0), build_prompt(ty0, B0, qk0)
tA = tokenizer(pA, return_tensors="pt")["input_ids"][0]
tB = tokenizer(pB, return_tensors="pt")["input_ids"][0]
posA, posB = identity_positions(pA), identity_positions(pB)
assert len(tA) == len(tB), (len(tA), len(tB))
assert posA == posB, (posA, posB)
diff = (tA != tB).nonzero().flatten().tolist()
print("Example prompt:\n" + pA)
print("\nTokens differing A vs B at positions:", diff, "| detected identity positions:", posA)
assert set(diff).issubset(set(posA)), "differing token is not at an identity position!"
for pos in posA:
    print(f"  pos {pos}: A={tokenizer.decode(tA[pos])!r}  B={tokenizer.decode(tB[pos])!r}")


## 3. Load model + the v2 patching machinery

Patch: `x_patched = x_base + α·(donor − base)` at the identity positions, per head.
α=1 = pure swap (instrument); α=5 = amplified (steering-style comparison).
The donor is taken from a forward pass of prompt B at the same positions.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

_forbidden = {(STAR_LAYER, STAR_HEAD), (SECOND_LAYER, 14), (11, 19)}
rng_ctrl = np.random.default_rng(RANDOM_SEED + 7)
while True:
    RAND_HEAD = (int(rng_ctrl.integers(0, model.config.num_hidden_layers)),
                 int(rng_ctrl.integers(0, NUM_HEADS)))
    if RAND_HEAD not in _forbidden:
        break
print("Random control head:", RAND_HEAD)

CAPTURE_LAYERS = sorted({STAR_LAYER, SECOND_LAYER, RAND_HEAD[0]})

_donor_capture = {}   # layer -> {pos: vec4096} (o_proj input)
_capture_positions = []
_active_patch = {}    # layer -> list of (pos, heads, alpha, donor_vec4096)

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            # batch-general: store ALL items in the batch, not just index 0.
            # The consumer (donors[...]) always indexes explicitly with [b] -> still 1D per item,
            # so Pass 2 (which always has B=1) needs no change at all.
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().float().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                for h in heads:
                    s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                    x[0, pos, s] = x[0, pos, s] + alpha * (d[s] - x[0, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

handles = [model.model.layers[L].self_attn.o_proj.register_forward_pre_hook(_oproj_prehook(L))
           for L in CAPTURE_LAYERS]
print("Hooks at layers", CAPTURE_LAYERS)

@torch.no_grad()
def forward_pred(prompt, n_opt, capture_positions=None, patch_spec=None):
    """patch_spec: dict layer -> list of (pos, heads, alpha, donor_vec)."""
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1, :]
    _active_patch.clear()
    _capture_positions = []
    selected = logits[LETTER_IDS[:n_opt]].float()
    return torch.softmax(selected, dim=0).cpu().numpy()

@torch.no_grad()
def forward_pred_batch(prompts, n_opt, capture_positions):
    """Batched version of forward_pred -- ONLY for baseline extraction (without patch_spec).
    Precondition: all prompts in `prompts` have the same TOKEN length (checked by the caller).
    """
    global _capture_positions
    _capture_positions = capture_positions
    _active_patch.clear()
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    logits = model(**inputs).logits[:, -1, :]
    _capture_positions = []
    selected = logits[:, LETTER_IDS[:n_opt]].float()
    return torch.softmax(selected, dim=1).cpu().numpy()  # [B, n_opt]


## 4. Pass 1 — baseline + donor at the identity positions (BATCHED)

Why the old loop was slow: every prompt ran ONE BY ONE (batch=1) -- the GPU
was indeed used, but the work per call is tiny compared with the surrounding
Python/tokenizer/hook overhead, so the GPU idles a lot waiting its turn.

Speed-up: all groups (`gk`) that share the SAME QUESTION (`qk`)
have prompts of IDENTICAL TOKEN LENGTH (they differ only in the 1 letter token of the
identity answer) -- so they can be merged into ONE batch, one forward pass, instead of
one at a time. There is a safety check: if some prompt turns out to have a different length
(it should not, but it is checked so nothing goes wrong silently), that group falls back
to the old path (one at a time) -- never silently wrong.


In [ ]:
BASELINE_BATCH_SIZE = 16  # safe on T4 VRAM for short prompts; lower it if OOM

baseline_pred = {}
donors = {}      # (ty, gk, qk) -> {layer: {pos: vec}}
id_pos = {}      # (ty, qk) -> positions (same for all cells of a type, per question)

for ty in TYPES_RUN:
    p = plan[ty]
    # group the not-yet-processed (gk, qk) by qk -- this is the key to batching
    by_qk = {}
    for (gk, qk) in p["needed"]:
        if (ty, gk, qk) in donors:
            continue
        by_qk.setdefault(qk, []).append(gk)

    for qk, gks in tqdm(by_qk.items(), desc=f"baseline {ty}"):
        prompts = [build_prompt(ty, gk, qk) for gk in gks]
        if (ty, qk) not in id_pos:
            id_pos[(ty, qk)] = identity_positions(prompts[0])
        positions = id_pos[(ty, qk)]
        n_opt = len(qmeta[qk][2])

        # safety check: all prompts in this group must have the same token length
        lens = [len(tokenizer.encode(pr, add_special_tokens=False)) for pr in prompts]

        if len(set(lens)) == 1:
            # fast path: merge all gk into one batch (chunked to avoid OOM)
            for start in range(0, len(gks), BASELINE_BATCH_SIZE):
                batch_gks = gks[start:start + BASELINE_BATCH_SIZE]
                batch_prompts = prompts[start:start + BASELINE_BATCH_SIZE]
                preds = forward_pred_batch(batch_prompts, n_opt, positions)
                for b, gk in enumerate(batch_gks):
                    baseline_pred[(gk, qk)] = preds[b]
                    donors[(ty, gk, qk)] = {
                        L: {pos: _donor_capture[L][pos][b].clone() for pos in positions}
                        for L in CAPTURE_LAYERS
                    }
        else:
            # fallback: differing token lengths (should not happen) -> one at a time, safe
            print(f"  [{ty} / {qk}] token lengths not uniform {set(lens)} -> per-item fallback")
            for gk, pr in zip(gks, prompts):
                pred = forward_pred(pr, n_opt, capture_positions=positions)
                baseline_pred[(gk, qk)] = pred
                donors[(ty, gk, qk)] = {
                    L: {pos: _donor_capture[L][pos][0].clone() for pos in positions}
                    for L in CAPTURE_LAYERS
                }

print(f"{len(baseline_pred)} baselines done.")


## 5. Pass 2 — patch conditions at the identity positions

In [ ]:
ALL_HEADS = list(range(NUM_HEADS))

def make_spec(ty, B, qk, layer_heads_alpha):
    """layer_heads_alpha: list of (layer, heads, alpha). Donor = B's."""
    positions = id_pos[(ty, qk)]
    spec = {}
    for (L, heads, alpha) in layer_heads_alpha:
        spec.setdefault(L, [])
        for pos in positions:
            spec[L].append((pos, heads, alpha, donors[(ty, B, qk)][L][pos]))
    return spec

CONDITIONS = {
    "patch_L11_all32":    [(STAR_LAYER, ALL_HEADS, 1.0)],
    "patch_L11H16":       [(STAR_LAYER, [STAR_HEAD], 1.0)],
    "patch_L11L18_all":   [(STAR_LAYER, ALL_HEADS, 1.0), (SECOND_LAYER, ALL_HEADS, 1.0)],
    "patch_L11_all32_x5": [(STAR_LAYER, ALL_HEADS, 5.0)],
    "patch_randhead":     [(RAND_HEAD[0], [RAND_HEAD[1]], 1.0)],
}

def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for pi, (A, B) in enumerate(tqdm(p["pairs"], desc=f"patch {ty}")):
        for qk in p["pair_questions"][(A, B)]:
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(ty, A, qk)
            base = dict(attr_type=ty, pair=f"{A} -> {B}", qkey=qk,
                        wd_A_to_realA=wd(predA, realA, ordinal),
                        wd_A_to_realB=wd(predA, realB, ordinal),
                        wd_B_to_realB=wd(predB, realB, ordinal),
                        wd_predA_predB=wd(predA, predB, ordinal),
                        real_wd_AB=real_wd(A, B, qk))
            for cond, lha in CONDITIONS.items():
                pp = forward_pred(prompt_A, n_opt, patch_spec=make_spec(ty, B, qk, lha))
                rows.append(dict(base, condition=cond,
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))
            if pi < N_SELF_PATCH:
                spec = make_spec(ty, A, qk, [(STAR_LAYER, ALL_HEADS, 1.0)])
                pp = forward_pred(prompt_A, n_opt, patch_spec=spec)
                rows.append(dict(base, condition="self_patch",
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))

res = pd.DataFrame(rows)
res["shift_to_realB"] = res["wd_A_to_realB"] - res["wd_patch_to_realB"]
res["shift_to_predB"] = res["wd_predA_predB"] - res["wd_patch_to_predB"]  # mechanical movement toward B's prediction
res.to_csv(os.path.join(OUT_DIR, "patching_v2_rows.csv"), index=False)
print(res.shape, "-> patching_v2_rows.csv")


## 6. Recap + statistical test

Two scores: `shift_to_realB` (fidelity — moving closer to B's REAL distribution) and
`shift_to_predB` (mechanical — moving closer to B's PREDICTION; this checks whether the patch
"works" at all). A patch may work mechanically yet not be faithful —
that separation is exactly what we want.


In [ ]:
summary_rows = []
print("=" * 110)
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty]
    ceiling = (r_ty.drop_duplicates(["pair", "qkey"])["wd_A_to_realB"]
               - r_ty.drop_duplicates(["pair", "qkey"])["wd_B_to_realB"]).mean()
    print(f"\n{ty} — full-swap ceiling (mean): {ceiling:+.4f}")
    rand = r_ty[r_ty["condition"] == "patch_randhead"].set_index(["pair", "qkey"])["shift_to_realB"]
    for cond in list(CONDITIONS) + ["self_patch"]:
        s_all = r_ty[r_ty["condition"] == cond].set_index(["pair", "qkey"])
        if len(s_all) == 0:
            continue
        s, sm = s_all["shift_to_realB"], s_all["shift_to_predB"]
        p_vs_rand = np.nan
        if cond not in ("patch_randhead", "self_patch"):
            joined = pd.concat([s, rand], axis=1, keys=["c", "r"]).dropna()
            if len(joined) > 10 and not np.allclose(joined["c"], joined["r"]):
                p_vs_rand = float(wilcoxon(joined["c"], joined["r"]).pvalue)
        summary_rows.append(dict(attr_type=ty, condition=cond, n=len(s),
                                 ceiling_full_swap=float(ceiling),
                                 mean_shift_to_realB=float(s.mean()),
                                 pct_positive=float((s > 0).mean()),
                                 mean_shift_to_predB=float(sm.mean()),
                                 p_wilcoxon_vs_random=p_vs_rand))
        p_str = "-" if np.isnan(p_vs_rand) else f"{p_vs_rand:.4f}"
        print(f"  {cond:20s} n={len(s):4d} | shift->realB={s.mean():+.4f} (>0: {(s>0).mean():.0%}) | "
              f"shift->predB={sm.mean():+.4f} | p vs random: {p_str}")

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUT_DIR, "patching_v2_summary.csv"), index=False)

print("\nBaseline context per type:")
for ty in TYPES_RUN:
    b = res[res["attr_type"] == ty].drop_duplicates(["pair", "qkey"])
    print(f"{ty:16s} WD(predA,predB)={b['wd_predA_predB'].mean():.4f} | "
          f"WD(predA,realB)={b['wd_A_to_realB'].mean():.4f} | "
          f"WD(predB,realB)={b['wd_B_to_realB'].mean():.4f} | "
          f"real WD between groups={b['real_wd_AB'].mean():.4f}")


## How to read the results

Order of checks:

1. **Does the patch work mechanically?** `shift_to_predB` for `patch_L11_all32` must be > 0
   and far above `patch_randhead`. If that is zero too -> identity is NOT carried
   through the L11 attention output at that position (the information travels another route/MLP).
2. **Working AND faithful?** `shift_to_realB` > 0, p < 0.05 vs random -> L11 has a
   demographically correct causal role. Compare the magnitude with
   `ceiling_full_swap` (what % of the ceiling is reached).
3. `patch_L11H16` vs `patch_L11_all32`: is 1 head enough or are 32 coordinated heads needed?
4. `x5` vs `x1`: if only x5 moves -> the effect needs amplification
   (in line with llm-opinions, which uses a 5-15x clamp) — the output is there but "damped".
5. `self_patch` must be ~0; `randhead` must be ~0.
6. The context line: if `WD(predA,predB)` stays small even with the QA prompt +
   the largest-real-WD questions -> output insensitivity (finding 07 D1-D2)
   is confirmed in their prompt format too. That is an important finding in its own right.

**Download:** `patching_v2_rows.csv` + `patching_v2_summary.csv` ->
`results/06_patching_v2/`. Results -> finding 08.
